In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SilverToGold") \
    .config(
        "spark.hadoop.hive.metastore.uris",
        "thrift://docker-hive-hive-metastore-1:9083"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

In [12]:
silver_path = "hdfs://docker-hive-namenode-1:8020/data-lake/silver/transaksi"

silver_df = spark.read.parquet(silver_path)

silver_df.show()
silver_df.printSchema()

+---+----------+------+----------+--------+------+-------+-------+
| id|   tanggal|produk|  kategori|    kota|jumlah|  harga|revenue|
+---+----------+------+----------+--------+------+-------+-------+
|  1|2026-05-01|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  2|2026-05-01| Mouse|Elektronik| Bandung|     2| 150000| 300000|
|  3|2026-05-02|  Buku|   Edukasi|Surabaya|     3|  75000| 225000|
|  4|2026-05-02|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  5|2026-05-03|Pulpen|   Edukasi|  Kediri|    10|   5000|  50000|
|  6|2026-05-03|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  7|2026-05-04| Mouse|Elektronik| Bandung|     1| 150000| 150000|
|  8|2026-05-04|  Buku|   Edukasi|Surabaya|     2|  75000| 150000|
+---+----------+------+----------+--------+------+-------+-------+

root
 |-- id: integer (nullable = true)
 |-- tanggal: date (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |--

In [13]:
from pyspark.sql.functions import sum

gold_revenue_produk = silver_df.groupBy(
    "produk", "kategori"
).agg(
    sum("revenue").alias("total_revenue")
)

gold_revenue_produk.show()

+------+----------+-------------+
|produk|  kategori|total_revenue|
+------+----------+-------------+
| Mouse|Elektronik|       450000|
|Pulpen|   Edukasi|        50000|
|  Buku|   Edukasi|       375000|
|Laptop|Elektronik|     24000000|
+------+----------+-------------+



In [14]:
gold_revenue_kota = silver_df.groupBy(
    "kota"
).agg(
    sum("revenue").alias("total_revenue")
)

gold_revenue_kota.show()

+--------+-------------+
|    kota|total_revenue|
+--------+-------------+
|  Kediri|        50000|
| Jakarta|     24000000|
|Surabaya|       375000|
| Bandung|       450000|
+--------+-------------+



In [15]:
gold_revenue_harian = silver_df.groupBy(
    "tanggal"
).agg(
    sum("revenue").alias("total_revenue")
).orderBy("tanggal")

gold_revenue_harian.show()

+----------+-------------+
|   tanggal|total_revenue|
+----------+-------------+
|2026-05-01|      8300000|
|2026-05-02|      8225000|
|2026-05-03|      8050000|
|2026-05-04|       300000|
+----------+-------------+



In [16]:
gold_revenue_produk.write.mode("overwrite").parquet(
    "hdfs://docker-hive-namenode-1:8020/data-lake/gold/revenue_produk"
)

gold_revenue_kota.write.mode("overwrite").parquet(
    "hdfs://docker-hive-namenode-1:8020/data-lake/gold/revenue_kota"
)

gold_revenue_harian.write.mode("overwrite").parquet(
    "hdfs://docker-hive-namenode-1:8020/data-lake/gold/revenue_harian"
)

In [17]:
gold_revenue_produk.write.mode("overwrite").saveAsTable(
    "warehouse.gold_revenue_produk"
)

gold_revenue_kota.write.mode("overwrite").saveAsTable(
    "warehouse.gold_revenue_kota"
)

gold_revenue_harian.write.mode("overwrite").saveAsTable(
    "warehouse.gold_revenue_harian"
)

In [18]:
spark.sql("SHOW TABLES IN warehouse").show(truncate=False)

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|warehouse|dim_produk         |false      |
|warehouse|fact_sales         |false      |
|warehouse|gold_revenue_harian|false      |
|warehouse|gold_revenue_kota  |false      |
|warehouse|gold_revenue_produk|false      |
|warehouse|revenue_kategori   |false      |
|warehouse|revenue_produk     |false      |
+---------+-------------------+-----------+



In [19]:
spark.sql("""
SELECT *
FROM warehouse.gold_revenue_produk
""").show()

+------+----------+-------------+
|produk|  kategori|total_revenue|
+------+----------+-------------+
| Mouse|Elektronik|       450000|
|Pulpen|   Edukasi|        50000|
|  Buku|   Edukasi|       375000|
|Laptop|Elektronik|     24000000|
+------+----------+-------------+

